# 13 — V4 Control Failure Diagnostics

## Goal
Do **not** create a new winner on the consumed 101-pair DEV corpus.

Instead:
1. Identify where Production Control is uniquely right/wrong versus chronology nuisance.
2. Diagnose which astrology feature families carry incremental signal.
3. Test whether adding each family to Control improves diagnostic CV.
4. Produce one evidence-based redesign direction for the next fresh confirmation DEV.

This notebook is **diagnostic only**. No sealed holdout is opened and no architecture can be promoted from this run.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_V4_CONTROL_FAILURE_DIAGNOSTICS_20260816"
SEED = 20260816
REPEATS = 10
FOLDS = 5

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p] + list(p.parents):
        if (c / "saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside the Chartpalja saju repo.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

ROOT = find_repo_root()
SRC = ROOT / "research/ml/artifacts/v4_unified_dev_3arch_tournament"
OUT = ROOT / "research/ml/artifacts/v4_control_failure_diagnostics"
OUT.mkdir(parents=True, exist_ok=True)

DECISION = SRC / "V4_UNIFIED_DEV_3ARCH_TOURNAMENT_DECISION.json"
PAIR_DIFF = SRC / "V4_UNIFIED_DEV_pair_diff_feature_table.csv"
OOF = SRC / "V4_UNIFIED_DEV_OOF_pair_scores.csv"
MANIFEST = SRC / "V4_UNIFIED_DEV_FEATURE_GENERATION_MANIFEST.json"

for p in [DECISION, PAIR_DIFF, OOF, MANIFEST]:
    if not p.exists():
        raise FileNotFoundError(p)

with open(DECISION, encoding="utf-8") as f:
    decision = json.load(f)
with open(MANIFEST, encoding="utf-8") as f:
    manifest = json.load(f)

pair_df = pd.read_csv(PAIR_DIFF)
oof = pd.read_csv(OOF)

assert decision["status"] == "V4_UNIFIED_DEV_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE"
assert decision["winner"] is None
assert len(pair_df) == 101
assert pair_df.subject_id.nunique() == 101
assert manifest["n_subjects"] == 101
assert manifest["n_year_rows"] == 202
assert all(v is False for v in decision["holdout_integrity"].values())

print("Preflight PASS")
print("pair_df:", pair_df.shape)
print("oof:", oof.shape)


Preflight PASS
pair_df: (101, 440)
oof: (7070, 8)


## 1. Control vs chronology nuisance: where does each actually win?

In [2]:

# Average repeated OOF correctness to one row per pair/model.
avg = (
    oof.groupby(["pair_id","subject_id","axis","positive_earlier_calc","model"])["correct"]
    .mean()
    .reset_index()
)

wide = avg.pivot_table(
    index=["pair_id","subject_id","axis","positive_earlier_calc"],
    columns="model",
    values="correct"
).reset_index()

for c in ["CONTROL_FIXED","NUISANCE_AGE_AXIS"]:
    assert c in wide.columns

# These fixed/OOF diagnostics may be fractional only for fitted nuisance due to repeated folds.
wide["control_correct"] = wide["CONTROL_FIXED"]
wide["nuisance_correct"] = wide["NUISANCE_AGE_AXIS"]

wide["control_only_advantage"] = wide["control_correct"] - wide["nuisance_correct"]
wide["nuisance_only_advantage"] = wide["nuisance_correct"] - wide["control_correct"]

slice_rows = []
for axis in ["ALL"] + sorted(wide.axis.unique()):
    x = wide if axis == "ALL" else wide[wide.axis == axis]
    for chrono in ["ALL","POS_EARLIER","POS_LATER"]:
        y = x
        if chrono == "POS_EARLIER":
            y = x[x.positive_earlier_calc.astype(bool)]
        elif chrono == "POS_LATER":
            y = x[~x.positive_earlier_calc.astype(bool)]
        if len(y) == 0:
            continue
        slice_rows.append({
            "axis": axis,
            "chronology": chrono,
            "n_pairs": len(y),
            "control_accuracy": float(y.control_correct.mean()),
            "nuisance_accuracy": float(y.nuisance_correct.mean()),
            "control_minus_nuisance": float(
                (y.control_correct-y.nuisance_correct).mean()
            ),
        })

slices = pd.DataFrame(slice_rows)
slices.to_csv(OUT / "V4_CONTROL_VS_NUISANCE_SLICES.csv", index=False)
display(slices)


,axis,chronology,n_pairs,control_accuracy,nuisance_accuracy,control_minus_nuisance
0,ALL,ALL,101,0.554455,0.662376,-0.107921
1,ALL,POS_EARLIER,57,0.552632,0.596491,-0.043860
2,ALL,POS_LATER,44,0.556818,0.747727,-0.190909
3,COMPETITIVE,ALL,31,0.548387,0.416129,0.132258
4,COMPETITIVE,POS_EARLIER,18,0.555556,0.000000,0.555556
5,COMPETITIVE,POS_LATER,13,0.538462,0.992308,-0.453846
6,PROJECT,ALL,25,0.620000,0.800000,-0.180000
7,PROJECT,POS_EARLIER,5,0.800000,0.000000,0.800000
8,PROJECT,POS_LATER,20,0.575000,1.000000,-0.425000
9,STATUS,ALL,45,0.522222,0.755556,-0.233333


## 2. Feature-family inventory

The goal is not to select a winner here. We only ask: **which representation families contain useful incremental information?**


In [3]:

diff_cols = [c for c in pair_df.columns if c.startswith("diff__")]
astro_cols = [c for c in diff_cols if not c.startswith("diff__baseline__")]
control_col = "diff__baseline__control_score"
assert control_col in pair_df.columns

nuisance_cols = sorted([c for c in pair_df.columns if c.startswith("nuisance__")])
assert nuisance_cols

def family_of(col):
    # diff__tg10__... -> tg10
    parts = col.split("__")
    return parts[1] if len(parts) >= 3 else "UNKNOWN"

families = {}
for c in astro_cols:
    families.setdefault(family_of(c), []).append(c)

family_inventory = pd.DataFrame([
    {"family": k, "n_features": len(v)}
    for k,v in sorted(families.items())
]).sort_values("n_features", ascending=False)

family_inventory.to_csv(OUT / "V4_FEATURE_FAMILY_INVENTORY.csv", index=False)
display(family_inventory)


,family,n_features
0,basic,166
7,tgxstrength,60
4,shinsal,43
6,tg10,40
8,unseong,28
3,relation,21
9,yongshin,16
2,hidden,10
5,tengod,10
1,engine,8


## 3. Diagnostic CV helpers

In [4]:

def symmetric_fit(train, cols, C=0.3):
    X = train[cols].to_numpy(dtype=float)
    X_aug = np.vstack([X, -X])
    y_aug = np.concatenate([
        np.ones(len(X), dtype=int),
        np.zeros(len(X), dtype=int),
    ])
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=float(C),
            solver="liblinear",
            random_state=SEED,
            max_iter=3000,
        ))
    ])
    model.fit(X_aug, y_aug)
    return model

def correct_from_model(model, test, cols):
    X = test[cols].to_numpy(dtype=float)
    score = model.decision_function(X)
    return (score > 0).astype(float)

def make_splits(frame, seed):
    meta = frame[["subject_id","axis","positive_earlier_calc"]].copy()
    assert meta.subject_id.nunique() == len(meta)
    meta["stratum"] = (
        meta.axis.astype(str) + "__" +
        meta.positive_earlier_calc.astype(int).astype(str)
    )
    counts = meta.stratum.value_counts()
    assert counts.min() >= FOLDS, counts.to_dict()

    skf = StratifiedKFold(
        n_splits=FOLDS, shuffle=True, random_state=int(seed)
    )
    X = np.zeros((len(meta),1))
    y = meta.stratum.to_numpy()

    out = []
    for tr, te in skf.split(X,y):
        out.append((
            meta.iloc[tr].subject_id.tolist(),
            meta.iloc[te].subject_id.tolist()
        ))
    return out

def cv_eval(cols, label):
    rows = []
    for rep in range(REPEATS):
        for fold, (train_ids, test_ids) in enumerate(make_splits(
            pair_df, SEED + 1000*rep
        )):
            train = pair_df[pair_df.subject_id.isin(train_ids)]
            test = pair_df[pair_df.subject_id.isin(test_ids)]
            assert len(test) > 0

            model = symmetric_fit(train, cols)
            correct = correct_from_model(model, test, cols)

            for (_, r), c in zip(test.iterrows(), correct):
                rows.append({
                    "model": label,
                    "repeat": rep,
                    "fold": fold,
                    "pair_id": r.pair_id,
                    "subject_id": r.subject_id,
                    "axis": r.axis,
                    "positive_earlier_calc": bool(r.positive_earlier_calc),
                    "correct": float(c),
                })
    return pd.DataFrame(rows)

print("CV helpers ready")


CV helpers ready


## 4. Family-only and incremental-to-Control diagnostics

For every feature family, run three **diagnostic** models:

- `FAMILY_ONLY`
- `NUISANCE + FAMILY`
- `CONTROL + FAMILY`

This is intentionally post-hoc exploration on consumed DEV and **cannot promote a production candidate**.


In [5]:

diag_parts = []

# Reference fitted models in identical CV mechanics.
diag_parts.append(cv_eval(nuisance_cols, "NUISANCE_REFIT"))
diag_parts.append(cv_eval([control_col], "CONTROL_REFIT"))

for family, cols in sorted(families.items()):
    diag_parts.append(cv_eval(cols, "FAMILY_ONLY__" + family))
    diag_parts.append(cv_eval(
        sorted(set(nuisance_cols + cols)),
        "NUISANCE_PLUS__" + family
    ))
    diag_parts.append(cv_eval(
        sorted(set([control_col] + cols)),
        "CONTROL_PLUS__" + family
    ))
    print("finished:", family, len(cols))

diag_oof = pd.concat(diag_parts, ignore_index=True)
diag_oof.to_csv(OUT / "V4_DIAGNOSTIC_FAMILY_OOF.csv", index=False)

summary = (
    diag_oof.groupby(["model","repeat"])["correct"].mean()
    .reset_index()
    .groupby("model")
    .agg(
        mean_accuracy=("correct","mean"),
        p10=("correct",lambda x: float(np.quantile(x,.10))),
        min_accuracy=("correct","min"),
        std=("correct","std"),
    )
    .reset_index()
)

summary.to_csv(OUT / "V4_DIAGNOSTIC_FAMILY_SUMMARY.csv", index=False)
display(summary.sort_values("mean_accuracy",ascending=False))


finished: basic 166
finished: engine 8
finished: hidden 10
finished: relation 21
finished: shinsal 43
finished: tengod 10
finished: tg10 40
finished: tgxstrength 60
finished: unseong 28
finished: yongshin 16


,model,mean_accuracy,p10,min_accuracy,std
31,NUISANCE_REFIT,0.662376,0.662376,0.653465,0.003131
23,NUISANCE_PLUS__hidden,0.662376,0.662376,0.653465,0.003131
22,NUISANCE_PLUS__engine,0.662376,0.662376,0.653465,0.003131
30,NUISANCE_PLUS__yongshin,0.656436,0.631683,0.613861,0.022408
24,NUISANCE_PLUS__relation,0.625743,0.584158,0.584158,0.034865
26,NUISANCE_PLUS__tengod,0.612871,0.593069,0.584158,0.017742
25,NUISANCE_PLUS__shinsal,0.610891,0.573267,0.564356,0.032015
29,NUISANCE_PLUS__unseong,0.592079,0.534653,0.534653,0.033266
21,NUISANCE_PLUS__basic,0.582178,0.550495,0.514851,0.031587
28,NUISANCE_PLUS__tgxstrength,0.581188,0.544554,0.544554,0.032353


## 5. Incremental family ranking

In [6]:

s = summary.set_index("model")

rows = []
for family in sorted(families):
    fam = "FAMILY_ONLY__" + family
    nc = "NUISANCE_PLUS__" + family
    cc = "CONTROL_PLUS__" + family

    rows.append({
        "family": family,
        "n_features": len(families[family]),
        "family_only": float(s.loc[fam,"mean_accuracy"]),
        "nuisance_plus_family": float(s.loc[nc,"mean_accuracy"]),
        "delta_vs_nuisance_refit": float(
            s.loc[nc,"mean_accuracy"] - s.loc["NUISANCE_REFIT","mean_accuracy"]
        ),
        "control_plus_family": float(s.loc[cc,"mean_accuracy"]),
        "delta_vs_control_refit": float(
            s.loc[cc,"mean_accuracy"] - s.loc["CONTROL_REFIT","mean_accuracy"]
        ),
        "control_plus_family_p10": float(s.loc[cc,"p10"]),
    })

ranking = pd.DataFrame(rows).sort_values(
    ["delta_vs_control_refit","delta_vs_nuisance_refit"],
    ascending=False
).reset_index(drop=True)

ranking.to_csv(
    OUT / "V4_FEATURE_FAMILY_INCREMENTAL_DIAGNOSTICS.csv",
    index=False
)
display(ranking)


,family,n_features,family_only,nuisance_plus_family,delta_vs_nuisance_refit,control_plus_family,delta_vs_control_refit,control_plus_family_p10
0,engine,8,0.000000,0.662376,0.000000,0.563366,0.000000,0.551485
1,hidden,10,0.000000,0.662376,0.000000,0.563366,0.000000,0.551485
2,basic,166,0.559406,0.582178,-0.080198,0.559406,-0.003960,0.513861
3,yongshin,16,0.548515,0.656436,-0.005941,0.538614,-0.024752,0.514851
4,shinsal,43,0.507921,0.610891,-0.051485,0.527723,-0.035644,0.485149
5,relation,21,0.531683,0.625743,-0.036634,0.516832,-0.046535,0.475248
6,tg10,40,0.475248,0.581188,-0.081188,0.475248,-0.088119,0.445545
7,tengod,10,0.490099,0.612871,-0.049505,0.449505,-0.113861,0.414851
8,tgxstrength,60,0.431683,0.581188,-0.081188,0.428713,-0.134653,0.385149
9,unseong,28,0.363366,0.592079,-0.070297,0.416832,-0.146535,0.365347


## 6. Axis stability for the most informative diagnostic families

In [7]:

top_families = ranking.head(min(5,len(ranking))).family.tolist()
wanted = ["CONTROL_REFIT","NUISANCE_REFIT"]
for fam in top_families:
    wanted += [
        "FAMILY_ONLY__"+fam,
        "CONTROL_PLUS__"+fam,
        "NUISANCE_PLUS__"+fam,
    ]

x = diag_oof[diag_oof.model.isin(wanted)]
axis_diag = (
    x.groupby(["model","axis"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)

axis_diag.to_csv(
    OUT / "V4_TOP_FAMILY_AXIS_DIAGNOSTICS.csv",
    index=False
)
display(axis_diag.pivot(index="model",columns="axis",values="accuracy"))


axis,COMPETITIVE,PROJECT,STATUS
model,,,
CONTROL_PLUS__basic,0.625806,0.612,0.484444
CONTROL_PLUS__engine,0.561290,0.624,0.531111
CONTROL_PLUS__hidden,0.561290,0.624,0.531111
CONTROL_PLUS__shinsal,0.538710,0.540,0.513333
CONTROL_PLUS__yongshin,0.487097,0.564,0.560000
CONTROL_REFIT,0.561290,0.624,0.531111
FAMILY_ONLY__basic,0.622581,0.620,0.482222
FAMILY_ONLY__engine,0.000000,0.000,0.000000
FAMILY_ONLY__hidden,0.000000,0.000,0.000000


## 7. Diagnostic decision for the next research step

In [8]:

top = ranking.iloc[0].to_dict()
positive_incremental = ranking[
    (ranking.delta_vs_control_refit > 0.0) |
    (ranking.delta_vs_nuisance_refit > 0.0)
].copy()

payload = {
    "version": "V4_CONTROL_FAILURE_DIAGNOSTIC_DECISION_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": "V4_CONTROL_FAILURE_DIAGNOSTICS_COMPLETE_REDESIGN_BEFORE_FRESH_CONFIRM_DEV",
    "source_tournament_status": decision["status"],
    "source_winner": decision["winner"],
    "corpus_role_after_this_run": "CONSUMED_ARCHITECTURE_DIAGNOSTIC_DEVELOPMENT",
    "n_pairs": 101,
    "top_incremental_family_diagnostic": {
        k: (float(v) if isinstance(v,(np.floating,float)) else
            int(v) if isinstance(v,(np.integer,int)) else v)
        for k,v in top.items()
    },
    "families_with_any_positive_incremental_direction": positive_incremental.family.tolist(),
    "rules": {
        "no_candidate_can_be_promoted_from_this_notebook": True,
        "no_holdout_opening_allowed": True,
        "do_not_add_posthoc_candidate_to_original_12_tournament": True,
        "next_architecture_may_use_these_diagnostics_for_design": True,
        "next_architecture_requires_fresh_confirmation_dev_before_gold_holdout": True,
    },
    "next_step": (
        "Use the control-vs-nuisance error slices and family incremental diagnostics to design "
        "one small, interpretable next-generation architecture. Freeze that architecture before "
        "collecting/scoring a fresh confirmation DEV corpus. Only if it beats Production Control "
        "there should a new GOLD holdout be constructed."
    ),
    "holdout_integrity": {
        "NEW_CONFIRM_loaded": False,
        "Validation_B_loaded": False,
        "Public_CHECK_loaded": False,
        "Public_FINAL_loaded": False,
    }
}

with open(
    OUT / "V4_CONTROL_FAILURE_DIAGNOSTIC_DECISION.json",
    "w", encoding="utf-8"
) as f:
    json.dump(payload,f,ensure_ascii=False,indent=2)

print(json.dumps(payload,ensure_ascii=False,indent=2))


{
  "version": "V4_CONTROL_FAILURE_DIAGNOSTIC_DECISION_V1",
  "notebook_version": "SAJU_ML_V4_CONTROL_FAILURE_DIAGNOSTICS_20260816",
  "created_at": "2026-08-16T22:05:56",
  "status": "V4_CONTROL_FAILURE_DIAGNOSTICS_COMPLETE_REDESIGN_BEFORE_FRESH_CONFIRM_DEV",
  "source_tournament_status": "V4_UNIFIED_DEV_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE",
  "source_winner": null,
  "corpus_role_after_this_run": "CONSUMED_ARCHITECTURE_DIAGNOSTIC_DEVELOPMENT",
  "n_pairs": 101,
  "top_incremental_family_diagnostic": {
    "family": "engine",
    "n_features": 8,
    "family_only": 0.0,
    "nuisance_plus_family": 0.6623762376237624,
    "delta_vs_nuisance_refit": 0.0,
    "control_plus_family": 0.5633663366336633,
    "delta_vs_control_refit": 0.0,
    "control_plus_family_p10": 0.5514851485148515
  },
  "families_with_any_positive_incremental_direction": [],
  "rules": {
    "no_candidate_can_be_promoted_from_this_notebook": true,
    "no_holdout_opening_allowed": true,
    "do_not_add_posthoc_c

## Send back

After **Kernel Restart → Run All**, send:

```text
V4_CONTROL_FAILURE_DIAGNOSTIC_DECISION.json
V4_FEATURE_FAMILY_INCREMENTAL_DIAGNOSTICS.csv
V4_CONTROL_VS_NUISANCE_SLICES.csv
V4_TOP_FAMILY_AXIS_DIAGNOSTICS.csv
V4_DIAGNOSTIC_FAMILY_SUMMARY.csv
```

Then the next artifact should be a **single redesigned architecture preregistration**, not another broad model tournament.
